# 0. Overview
Author: Darrin O'Brien, email: darrinobrien5@gmail.com

1. Fine-Tunes CLIP ViT-32 on the RESISC45 Dataset (https://huggingface.co/datasets/tanganke/resisc45)
2. Evaluates the performance of the fine-tuned model.

## 1. Quick Installs for Essential Libraries

In [ ]:
!pip install torch torchvision
!pip install fifty regex tqdm
!pip install git+https://github.com/openai/CLIP.git
!pip install pandas scipy
!pip install -U scikit-learn
!pip install --upgrade Pillow
!pip install -U datasets transformers

### 1. Runpod Only Installs

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

## 2. Importing Libraries

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import clip
from tqdm import tqdm
from PIL import ExifTags
import PIL
from datasets import load_dataset, load_from_disk
from PIL import Image
import io
import numpy as np
from transformers import get_cosine_schedule_with_warmup

## 3. Setting up Device and Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device) # https://github.com/openai/CLIP
clip_model = clip_model.float().to(device) # For fp-32 precision

## 4. Preparing Dataset

In [ ]:
# https://huggingface.co/datasets/tanganke/resisc45

# Comment out if you already have the preprocssed images

dataset = load_dataset('tanganke/resisc45', cache_dir="/workspace/.hf_cache")

split = dataset["train"].train_test_split(test_size=0.2, seed=66)

train = split["train"]
val = split["test"]
test = dataset["test"]

def transform_example(example):
    example["image"] = preprocess(example["image"]).numpy()
    return example

train = train.map(transform_example)
val = val.map(transform_example)
test = test.map(transform_example)

train.save_to_disk("/workspace/preprocessed/RESISC45/train_converted")
val.save_to_disk("/workspace/preprocessed/RESISC45/val_converted")
test.save_to_disk("/workspace/preprocessed/RESISC45/test_converted")


In [ ]:
# Loading Fully Processed
train = load_from_disk("/workspace/preprocessed/RESISC45/train_converted")
val = load_from_disk("/workspace/preprocessed/RESISC45/val_converted")
test = load_from_disk("/workspace/preprocessed/RESISC45/test_converted")

In [ ]:
train.set_format(type="python", columns=["image", "label"])
val.set_format(type="python", columns=["image", "label"])
test.set_format(type="python", columns=["image", "label"])

def clip_collate_fn(batch):
    images = np.stack([example["image"] for example in batch]).astype(np.float32)
    images = torch.from_numpy(images)
    labels = torch.tensor([example["label"] for example in batch], dtype=torch.long)
    
    return {
        "pixel_values": images.to(device),
        "labels": labels.to(device)
    }

train_loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=clip_collate_fn)
val_loader = DataLoader(val, batch_size=64, shuffle=False, collate_fn=clip_collate_fn)
test_loader = DataLoader(test, batch_size=64, shuffle=False, collate_fn=clip_collate_fn)

## 5. Fine-Tune Prep

In [ ]:
class CLIPClassifier(nn.Module):
  def __init__(self, clip_model, num_classes=45): # 45 classes in RESISC45 dataset
    super().__init__()
    self.clip = clip_model
    self.classifier = nn.Linear(self.clip.visual.output_dim, num_classes)

  def forward(self, images):
    image_features = self.clip.encode_image(images)
    logits = self.classifier(image_features)
    return logits

model = CLIPClassifier(clip_model=clip_model).to(device)
model = model.float()

In [ ]:
if device == "cpu":
  model = model.float()

optimizer = optim.Adam(model.parameters(), lr=1e-5)

criterion = nn.CrossEntropyLoss() 

EPOCHS = 20 # For fp-32
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * EPOCHS)

## 6. Fine-Tuning CLIP on Stanford Cars Dataset

In [ ]:

best_val_loss = float('inf')
best_epoch = -1

for epoch in range(EPOCHS):
  print(f"Epoch {epoch+1}/{EPOCHS} - Best Val Loss: {best_val_loss:.4f} (Epoch {best_epoch})")

  model.train()
  total_train_loss = 0
  train_steps = 0

  for batch in tqdm(train_loader, desc="Training"):
    optimizer.zero_grad()

    images = batch["pixel_values"]
    labels = batch["labels"]

    logits = model(images)
    loss = criterion(logits, labels)

    loss.backward()
    optimizer.step()

    total_train_loss += loss.item()
    train_steps += 1

  avg_train_loss = total_train_loss / train_steps

  # Validation
  model.eval()
  correct = 0
  total = 0
  total_val_loss = 0
  val_steps = 0

  with torch.no_grad():
    for batch in tqdm(val_loader, desc="Validation"):
      images = batch["pixel_values"]
      labels = batch["labels"]

      logits = model(images)
      loss = criterion(logits, labels)

      preds = torch.argmax(logits, dim=1)
      correct += (preds == labels).sum().item()
      total += labels.size(0)

      total_val_loss += loss.item()
      val_steps += 1

  avg_val_loss = total_val_loss / val_steps
  val_acc = correct / total

  print(f"[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

  if avg_val_loss < best_val_loss:
    best_val_loss = avg_val_loss
    best_epoch = epoch
    torch.save(model.state_dict(), "best_clip_RESISC45.pt")

  scheduler.step()

## 7. Testing Fine-Tuned Model

In [ ]:

base_CLIP, _ = clip.load("ViT-B/32", device=device)
base_CLIP = base_CLIP.float() # fp-32
model = CLIPClassifier(clip_model=base_CLIP).to(device)


best_CLIP, _ = clip.load("ViT-B/32", device=device)
best_CLIP = best_CLIP.float() # fp-32
best_CLIP_Cars = CLIPClassifier(clip_model=best_CLIP).to(device)
best_CLIP_Cars.load_state_dict(torch.load("best_clip_RESISC45.pt", map_location=device)) # map_location tells where to place the model's weights in memory

model.eval()
best_CLIP_Cars.eval()

total_test_loss_base = 0
total_base = 0
total_test_loss_best = 0
total_best = 0

correct_base = 0
correct_best = 0
total_samples = 0

with torch.no_grad():
  for batch in tqdm(test_loader, desc="Testing"):
    images = batch["pixel_values"]
    labels = batch["labels"]
    total_samples += labels.size(0)

    # Base model
    logits_base = model(images)
    loss_base = criterion(logits_base, labels)
    total_test_loss_base += loss_base.item()
    total_base += 1

    # Best model
    logits_best = best_CLIP_Cars(images)
    loss_best = criterion(logits_best, labels)
    total_test_loss_best += loss_best.item()
    total_best += 1

    # Classification Accuracy
    pred_base = logits_base.argmax(dim=1)
    pred_best = logits_best.argmax(dim=1)

    correct_base += (pred_base == labels).sum().item()
    correct_best += (pred_best == labels).sum().item()

avg_base_loss = total_test_loss_base / total_base
avg_best_loss = total_test_loss_best / total_best

accuracy_base = correct_base / total_samples
accuracy_best = correct_best / total_samples
print(f"\nAverage base loss: {avg_base_loss:.4f}, Base Accuracy: {accuracy_base:.4f}")
print(f"Average best loss: {avg_best_loss:.4f}, Best Accuracy: {accuracy_best:.4f}")

In [ ]:
# best_CLIP_RESISC45.pt
# [Epoch 4] Train Loss: 0.0352 | Validation Loss: 0.2938 | Validation Accuracy: 0.9259
# Average base loss: 3.8508, Base Accuracy: 0.0156
# Average best loss: 0.2961, Best Accuracy: 0.9270